# Coordinator NB04 — `CoordinatorReport`

Builds the full decision packet consumed by the LLM to generate actionable advice.

**Design principle**: Coordinator computes ALL numbers deterministically. LLM only narrates — never invents or modifies numbers.

```
CoordinatorSignal (NB03) → PairAnalysis (NB04) → CoordinatorReport → LLM prompt
```

**What `PairAnalysis` adds on top of `CoordinatorSignal`:**
- `vol_signal_norm` — sign-normalized: higher = more vol expected (stocktwits inverted)
- `estimated_vol_3d` — calibrated vol forecast in log-return units (OLS on training data)
- `conviction_score` — `tier_weight × direction_edge`
- `position_size_pct` — % of equity; discounted 25% in `high_attention` regime
- `sl_pct` / `tp_pct` — % from entry price: 1.5× / 2.5× expected move at the trade horizon
- `raw_signals` — all agent outputs for LLM context

**`CoordinatorReport` wraps all 4 `PairAnalysis` objects:**
- `ranked_pairs` — sorted by conviction descending
- `top_pick` — highest-conviction pair when `overall_action == "trade"`
- `overall_action` — `"trade"` | `"hold"` (HOLD gate: max conviction below threshold)
- `narrative_context` — pre-formatted dict for LLM prompt (no number synthesis needed)

**HOLD gate:**
- `max(conviction) < 0.02` → hold (nothing meaningful)
- `regime == high_attention` AND `max(conviction) < 0.05` → hold (high vol + weak signal)

In [25]:
from __future__ import annotations

import sys
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LinearRegression

ROOT = Path("..")
sys.path.insert(0, str(ROOT))

SIGNALS = ROOT / "data/processed/coordinator/signals_aligned.parquet"
df = pd.read_parquet(SIGNALS)
df["date"] = pd.to_datetime(df["date"])

print(f"{len(df):,} rows | {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Pairs: {sorted(df['pair'].unique())}")

4,172 rows | 2022-01-03 → 2025-12-31
Pairs: ['EURUSD', 'GBPUSD', 'USDCHF', 'USDJPY']


## 1. Re-define CoordinatorSignal + `coordinator_predict` (from NB03)

In [26]:
@dataclass
class CoordinatorSignal:
    pair: str
    date: pd.Timestamp
    vol_signal: float | None
    vol_source: str               # "stocktwits" | "geo" | "none"
    direction: int | None         # 1=up (base appreciates), 0=down, None=no data
    direction_source: str
    direction_horizon: str        # "1d" | "5d" | "none"
    direction_ic: float | None
    confidence_tier: str          # "high" | "medium" | "low" | "none"
    flat_reason: str | None
    regime: str                   # "high_attention" | "normal" | "unknown"


USDJPY_CONF_P75 = float(
    df[(df["pair"] == "USDJPY") & (df["date"] <= pd.Timestamp("2023-12-31"))]["tech_confidence"].quantile(0.75)
)

PAIR_MACRO_CONFIG: dict[str, tuple[str, str, float]] = {
    "GBPUSD": ("medium", "5d", 0.115),
    "EURUSD": ("low",    "5d", 0.060),
    "USDCHF": ("low",    "5d", 0.033),
    "USDJPY": ("low",    "5d", 0.058),
}
REGIME_ATTENTION_THRESHOLD = 1.0


def coordinator_predict(row: pd.Series) -> CoordinatorSignal:
    pair = str(row["pair"])
    date = pd.Timestamp(row["date"])

    stocktwits_val = row.get("usdjpy_stocktwits_vol_signal")
    if pair == "USDJPY" and pd.notna(stocktwits_val):
        vol_signal, vol_source = float(stocktwits_val), "stocktwits"
    else:
        geo_val = row.get("geo_bilateral_risk")
        vol_signal = float(geo_val) if pd.notna(geo_val) else None
        vol_source = "geo" if vol_signal is not None else "none"

    flat_reason: str | None = None
    if pair == "USDJPY":
        conf = row.get("tech_confidence")
        if pd.notna(conf) and float(conf) >= USDJPY_CONF_P75:
            direction, direction_source = int(row["tech_direction"]), "tech_usdjpy"
            direction_horizon, direction_ic, confidence_tier = "1d", None, "high"
        else:
            macro_dir = row.get("macro_direction")
            if pd.notna(macro_dir):
                direction = 1 if macro_dir == "up" else 0
                tier, horizon, ic_val = PAIR_MACRO_CONFIG["USDJPY"]
                direction_source, direction_horizon, direction_ic, confidence_tier = "macro_usdjpy", horizon, ic_val, tier
            else:
                direction, direction_source = None, "flat"
                direction_horizon, direction_ic, confidence_tier = "none", None, "none"
                flat_reason = "macro_direction unavailable"
    else:
        macro_dir = row.get("macro_direction")
        if pd.notna(macro_dir):
            direction = 1 if macro_dir == "up" else 0
            tier, horizon, ic_val = PAIR_MACRO_CONFIG[pair]
            direction_source = f"macro_{pair.lower()}"
            direction_horizon, direction_ic, confidence_tier = horizon, ic_val, tier
        else:
            direction, direction_source = None, "flat"
            direction_horizon, direction_ic, confidence_tier = "none", None, "none"
            flat_reason = "macro_direction unavailable"

    macro_z = row.get("macro_attention_zscore")
    regime = "high_attention" if pd.notna(macro_z) and float(macro_z) > REGIME_ATTENTION_THRESHOLD else (
        "normal" if pd.notna(macro_z) else "unknown"
    )

    return CoordinatorSignal(
        pair=pair, date=date,
        vol_signal=vol_signal, vol_source=vol_source,
        direction=direction, direction_source=direction_source,
        direction_horizon=direction_horizon, direction_ic=direction_ic,
        confidence_tier=confidence_tier, flat_reason=flat_reason, regime=regime,
    )


print(f"coordinator_predict() ready | USDJPY p75={USDJPY_CONF_P75:.6f}")

coordinator_predict() ready | USDJPY p75=0.088708


In [28]:
# Build coord DataFrame (NB03 batch run) for calibration
signals_list = [coordinator_predict(row) for _, row in df.iterrows()]
coord = pd.DataFrame({
    "date":              [s.date for s in signals_list],
    "pair":              [s.pair for s in signals_list],
    "vol_signal":        [s.vol_signal for s in signals_list],
    "vol_source":        [s.vol_source for s in signals_list],
    "vol_signal_norm":   [
        (-s.vol_signal if s.vol_source == "stocktwits" and s.vol_signal is not None else s.vol_signal)
        for s in signals_list
    ],
    "direction":         [s.direction for s in signals_list],
    "direction_source":  [s.direction_source for s in signals_list],
    "direction_horizon": [s.direction_horizon for s in signals_list],
    "direction_ic":      [s.direction_ic for s in signals_list],
    "confidence_tier":   [s.confidence_tier for s in signals_list],
    "regime":            [s.regime for s in signals_list],
}).merge(df[["date", "pair", "fwd_vol_3d", "fwd_ret_1d", "fwd_ret_5d"]], on=["date", "pair"], how="left")

TRAIN_CUTOFF = pd.Timestamp("2023-12-31")
train_all = coord.dropna(subset=["fwd_vol_3d", "vol_signal_norm"])

# Geo: OLS on training window (2022-2023)
geo_tr = train_all[(train_all["vol_source"] == "geo") & (train_all["date"] <= TRAIN_CUTOFF)]
lr_geo = LinearRegression().fit(geo_tr["vol_signal_norm"].values.reshape(-1,1), geo_tr["fwd_vol_3d"].values)
GEO_SLOPE     = float(lr_geo.coef_[0])
GEO_INTERCEPT = float(lr_geo.intercept_)

# StockTwits: started post-2023, calibrate on all available rows (sign-inverted → vol_signal_norm)
st_all = train_all[train_all["vol_source"] == "stocktwits"]
print(f"StockTwits rows available: {len(st_all)} | dates: {st_all['date'].min().date()} → {st_all['date'].max().date()}")
st_half = st_all[st_all["date"] <= pd.Timestamp("2024-12-31")]  # use 2024 as "in-sample" for ST
lr_st = LinearRegression().fit(st_half["vol_signal_norm"].values.reshape(-1,1), st_half["fwd_vol_3d"].values)
ST_SLOPE     = float(lr_st.coef_[0])
ST_INTERCEPT = float(lr_st.intercept_)

BASELINE_VOL = float(train_all[train_all["date"] <= TRAIN_CUTOFF]["fwd_vol_3d"].mean())

print(f"\nGeo calibration:        intercept={GEO_INTERCEPT:.6f}  slope={GEO_SLOPE:.6f}")
print(f"StockTwits calibration: intercept={ST_INTERCEPT:.6f}  slope={ST_SLOPE:.6f}")
print(f"Baseline vol (2022-2023 mean): {BASELINE_VOL:.6f}  (~{BASELINE_VOL*100:.3f}% daily)")

# Sanity: test-set rank IC for geo
test = coord[coord["date"] > TRAIN_CUTOFF].dropna(subset=["fwd_vol_3d", "vol_signal_norm"])
pred_geo = GEO_INTERCEPT + GEO_SLOPE * test.loc[test["vol_source"]=="geo", "vol_signal_norm"]
actual_geo = test.loc[test["vol_source"]=="geo", "fwd_vol_3d"]
r_geo, _ = stats.spearmanr(pred_geo, actual_geo)
st_test = coord[(coord["date"] > pd.Timestamp("2024-12-31"))].dropna(subset=["fwd_vol_3d", "vol_signal_norm"])
st_test = st_test[st_test["vol_source"] == "stocktwits"]
if len(st_test) > 10:
    r_st, _ = stats.spearmanr(ST_INTERCEPT + ST_SLOPE * st_test["vol_signal_norm"], st_test["fwd_vol_3d"])
    print(f"\nTest-set rank IC (estimated_vol_3d vs fwd_vol_3d):")
    print(f"  Geo (2024-2025):         {r_geo:+.4f}")
    print(f"  StockTwits (2025 only):  {r_st:+.4f}")

StockTwits rows available: 414 | dates: 2024-05-27 → 2025-12-26

Geo calibration:        intercept=0.004057  slope=0.002405
StockTwits calibration: intercept=0.006403  slope=0.004930
Baseline vol (2022-2023 mean): 0.004524  (~0.452% daily)

Test-set rank IC (estimated_vol_3d vs fwd_vol_3d):
  Geo (2024-2025):         +0.1162
  StockTwits (2025 only):  +0.2303


## 3. `PairAnalysis` + `CoordinatorReport` Dataclasses

In [29]:
@dataclass
class PairAnalysis:
    pair: str
    # Directional track
    direction: int | None           # 1=long base, 0=short base, None=no data
    direction_label: str            # e.g. "LONG GBPUSD"
    confidence_tier: str            # "high" | "medium" | "low" | "none"
    direction_source: str
    direction_horizon: str          # "1d" | "5d" | "none"
    direction_ic: float | None
    # Vol track
    vol_signal_norm: float | None   # sign-normalized (higher = more vol expected)
    vol_source: str
    estimated_vol_3d: float         # calibrated expected daily vol in log-return units
    regime: str
    # Trade parameters (deterministic)
    suggested_action: str           # "LONG" | "SHORT" | "FLAT"
    conviction_score: float         # tier_weight × direction_edge
    position_size_pct: float        # % of equity to risk
    sl_pct: float                   # % stop-loss distance from entry
    tp_pct: float                   # % take-profit distance from entry
    risk_reward_ratio: float        # tp_pct / sl_pct
    # All raw signal values for LLM context
    raw_signals: dict


@dataclass
class CoordinatorReport:
    date: pd.Timestamp
    pairs: list[PairAnalysis]
    ranked_pairs: list[str]         # sorted by conviction descending
    top_pick: str | None            # None when overall_action == "hold"
    overall_action: str             # "trade" | "hold"
    hold_reason: str | None
    global_regime: str
    narrative_context: dict         # pre-formatted packet for LLM


print("PairAnalysis + CoordinatorReport defined")

PairAnalysis + CoordinatorReport defined


## 4. Deterministic Computation — Conviction, Position Size, SL/TP, HOLD Gate

### Conviction score
`conviction = tier_weight × direction_edge`

| Source | Edge metric | Tier weight |
|--------|-------------|-------------|
| `tech_usdjpy` | 0.086 (accuracy − 0.5 = 0.586 − 0.5) | 1.0 (high) |
| `macro_gbpusd` | 0.115 (IC_5d) | 0.6 (medium) |
| `macro_*` | IC_5d (0.033–0.060) | 0.3 (low) |

### Position sizing
- Base: high=2%, medium=1%, low=0.5% of equity
- `high_attention` regime: ×0.75 (reduce for elevated vol)

### SL/TP
- `estimated_vol_3d` ≈ daily std of log returns
- 1d horizon: `expected_move = estimated_vol_3d`
- 5d horizon: `expected_move = estimated_vol_3d × √5`
- `SL = 1.5 × expected_move × 100` (as % of entry)
- `TP = 2.5 × expected_move × 100`

### HOLD gate
- `max(conviction) < 0.02` → hold unconditionally
- `regime == high_attention` AND `max(conviction) < 0.05` → hold

In [30]:
TIER_WEIGHTS    = {"high": 1.0, "medium": 0.6, "low": 0.3, "none": 0.0}
BASE_POSITION   = {"high": 2.0, "medium": 1.0, "low": 0.5, "none": 0.0}  # % of equity
TECH_EDGE       = 0.086    # 0.586 − 0.5: empirical accuracy edge on gated USDJPY days
SL_MULT         = 1.5      # SL = 1.5× expected move
TP_MULT         = 2.5      # TP = 2.5× expected move (R:R = 1.67)
REGIME_DISCOUNT = 0.75     # position scale-down in high_attention regime

HOLD_MIN_CONVICTION          = 0.02   # absolute floor — hold if nothing meaningful
HOLD_REGIME_CONVICTION_FLOOR = 0.05   # floor for high_attention regime


def compute_conviction(tier: str, direction_source: str, direction_ic: float | None) -> float:
    weight = TIER_WEIGHTS.get(tier, 0.0)
    edge = TECH_EDGE if direction_source == "tech_usdjpy" else (abs(direction_ic) if direction_ic else 0.0)
    return round(weight * edge, 4)


def compute_position_pct(tier: str, regime: str) -> float:
    size = BASE_POSITION.get(tier, 0.0)
    if regime == "high_attention":
        size *= REGIME_DISCOUNT
    return round(size, 4)


def estimate_vol_3d(vol_signal_norm: float | None, vol_source: str) -> float:
    if vol_signal_norm is None:
        return BASELINE_VOL
    if vol_source == "geo":
        return max(0.001, GEO_INTERCEPT + GEO_SLOPE * vol_signal_norm)
    if vol_source == "stocktwits":
        return max(0.001, ST_INTERCEPT + ST_SLOPE * vol_signal_norm)
    return BASELINE_VOL


def compute_sl_tp(estimated_vol: float, direction_horizon: str) -> tuple[float, float]:
    """Return (sl_pct, tp_pct) as % of entry price."""
    if direction_horizon == "1d":
        expected_move = estimated_vol
    elif direction_horizon == "5d":
        expected_move = estimated_vol * (5 ** 0.5)
    else:
        expected_move = estimated_vol * 2.0
    sl = round(SL_MULT * expected_move * 100, 4)
    tp = round(TP_MULT * expected_move * 100, 4)
    return sl, tp


def check_hold_gate(analyses: list[PairAnalysis], global_regime: str) -> tuple[str, str | None]:
    max_conv = max(a.conviction_score for a in analyses)
    if max_conv < HOLD_MIN_CONVICTION:
        return "hold", f"Max conviction {max_conv:.4f} < {HOLD_MIN_CONVICTION} — no meaningful signal"
    if global_regime == "high_attention" and max_conv < HOLD_REGIME_CONVICTION_FLOOR:
        return "hold", f"High attention regime + conviction {max_conv:.4f} < {HOLD_REGIME_CONVICTION_FLOOR}"
    return "trade", None


print("Computation functions defined")
print(f"\nConviction examples:")
for pair, (tier, _, ic) in PAIR_MACRO_CONFIG.items():
    src = "tech_usdjpy" if pair == "USDJPY" else f"macro_{pair.lower()}"
    c = compute_conviction(tier, src, ic if src != "tech_usdjpy" else None)
    print(f"  {src:<18}: conviction={c:.4f}  pos={compute_position_pct(tier, 'normal'):.1f}%")

Computation functions defined

Conviction examples:
  macro_gbpusd      : conviction=0.0690  pos=1.0%
  macro_eurusd      : conviction=0.0180  pos=0.5%
  macro_usdchf      : conviction=0.0099  pos=0.5%
  tech_usdjpy       : conviction=0.0258  pos=0.5%


## 5. `build_pair_analysis` + `build_coordinator_report`

In [31]:
def build_pair_analysis(sig: CoordinatorSignal, row: pd.Series) -> PairAnalysis:
    # Sign-normalize vol: stocktwits IC is negative → invert so higher = more vol
    if sig.vol_source == "stocktwits" and sig.vol_signal is not None:
        vol_signal_norm = -sig.vol_signal
    else:
        vol_signal_norm = sig.vol_signal

    est_vol = estimate_vol_3d(vol_signal_norm, sig.vol_source)

    if sig.direction == 1:
        direction_label, suggested_action = f"LONG {sig.pair}", "LONG"
    elif sig.direction == 0:
        direction_label, suggested_action = f"SHORT {sig.pair}", "SHORT"
    else:
        direction_label, suggested_action = f"FLAT {sig.pair}", "FLAT"

    conv = compute_conviction(sig.confidence_tier, sig.direction_source, sig.direction_ic)
    pos_pct = compute_position_pct(sig.confidence_tier, sig.regime) if sig.direction is not None else 0.0
    sl, tp = compute_sl_tp(est_vol, sig.direction_horizon)
    rr = round(tp / sl, 2) if sl > 0 else 0.0

    raw_signals = {
        "tech_direction":          row.get("tech_direction"),
        "tech_confidence":         row.get("tech_confidence"),
        "tech_vol_regime":         row.get("tech_vol_regime"),
        "geo_bilateral_risk":      row.get("geo_bilateral_risk"),
        "geo_risk_regime":         row.get("geo_risk_regime"),
        "macro_direction":         row.get("macro_direction"),
        "macro_confidence":        row.get("macro_confidence"),
        "macro_dominant_driver":   row.get("macro_dominant_driver"),
        "macro_carry_score":       row.get("macro_carry_score"),
        "macro_fundamental_score": row.get("macro_fundamental_score"),
        "macro_regime_score":      row.get("macro_regime_score"),
        "macro_surprise_score":    row.get("macro_surprise_score"),
        "macro_bias_score":        row.get("macro_bias_score"),
        "gdelt_tone_zscore":       row.get("gdelt_tone_zscore"),
        "macro_attention_zscore":  row.get("macro_attention_zscore"),
        "composite_stress_flag":   row.get("composite_stress_flag"),
        **({"usdjpy_stocktwits_vol_signal": row.get("usdjpy_stocktwits_vol_signal")}
           if sig.pair == "USDJPY" else {}),
    }

    return PairAnalysis(
        pair=sig.pair, direction=sig.direction, direction_label=direction_label,
        confidence_tier=sig.confidence_tier, direction_source=sig.direction_source,
        direction_horizon=sig.direction_horizon, direction_ic=sig.direction_ic,
        vol_signal_norm=vol_signal_norm, vol_source=sig.vol_source,
        estimated_vol_3d=est_vol, regime=sig.regime,
        suggested_action=suggested_action, conviction_score=conv,
        position_size_pct=pos_pct, sl_pct=sl, tp_pct=tp, risk_reward_ratio=rr,
        raw_signals=raw_signals,
    )


def build_narrative_context(
    analyses: list[PairAnalysis], date: pd.Timestamp,
    overall_action: str, hold_reason: str | None, global_regime: str,
) -> dict:
    ranked = sorted(analyses, key=lambda a: a.conviction_score, reverse=True)
    top = ranked[0] if overall_action == "trade" else None

    ctx: dict = {
        "date": str(date.date()),
        "overall_action": overall_action,
        "hold_reason": hold_reason,
        "global_regime": global_regime,
        "top_pick": top.pair if top else None,
    }

    if top:
        ctx["top_recommendation"] = {
            "pair":              top.pair,
            "action":            top.suggested_action,
            "confidence":        top.confidence_tier,
            "position_size_pct": top.position_size_pct,
            "sl_pct":            top.sl_pct,
            "tp_pct":            top.tp_pct,
            "risk_reward":       top.risk_reward_ratio,
            "horizon":           top.direction_horizon,
            "signal_source":     top.direction_source,
            "direction_ic":      top.direction_ic,
            "estimated_vol_pct": round(top.estimated_vol_3d * 100, 3),
            "vol_source":        top.vol_source,
        }

    ctx["all_pairs"] = [
        {
            "pair":              a.pair,
            "action":            a.suggested_action,
            "confidence":        a.confidence_tier,
            "conviction":        a.conviction_score,
            "horizon":           a.direction_horizon,
            "estimated_vol_pct": round(a.estimated_vol_3d * 100, 3),
            "vol_source":        a.vol_source,
            "regime":            a.regime,
            "macro_driver":      a.raw_signals.get("macro_dominant_driver"),
            "geo_risk_regime":   a.raw_signals.get("geo_risk_regime"),
            "macro_attention_z": a.raw_signals.get("macro_attention_zscore"),
            "gdelt_tone_z":      a.raw_signals.get("gdelt_tone_zscore"),
            "composite_stress":  a.raw_signals.get("composite_stress_flag"),
        }
        for a in ranked
    ]
    return ctx


def build_coordinator_report(date_rows: pd.DataFrame) -> CoordinatorReport:
    date = pd.Timestamp(date_rows["date"].iloc[0])
    analyses: list[PairAnalysis] = []
    for _, row in date_rows.iterrows():
        sig = coordinator_predict(row)
        analyses.append(build_pair_analysis(sig, row))

    global_regime = analyses[0].regime if analyses else "unknown"
    overall_action, hold_reason = check_hold_gate(analyses, global_regime)

    tradeable = [a for a in analyses if a.direction is not None]
    ranked_pairs = [a.pair for a in sorted(tradeable, key=lambda a: a.conviction_score, reverse=True)]
    top_pick = ranked_pairs[0] if ranked_pairs and overall_action == "trade" else None

    ctx = build_narrative_context(analyses, date, overall_action, hold_reason, global_regime)

    return CoordinatorReport(
        date=date, pairs=analyses, ranked_pairs=ranked_pairs,
        top_pick=top_pick, overall_action=overall_action, hold_reason=hold_reason,
        global_regime=global_regime, narrative_context=ctx,
    )


print("build_pair_analysis / build_coordinator_report defined")

build_pair_analysis / build_coordinator_report defined


## 6. Smoke Test — Single Date Report

In [32]:
import json

def print_report(report: CoordinatorReport) -> None:
    print(f"{'='*65}")
    print(f"CoordinatorReport  {report.date.date()}  |  regime={report.global_regime}")
    print(f"  overall_action: {report.overall_action.upper()}", end="")
    if report.hold_reason:
        print(f"  ({report.hold_reason})", end="")
    print()
    if report.top_pick:
        print(f"  top_pick: {report.top_pick}  |  ranked: {report.ranked_pairs}")
    print()
    for a in sorted(report.pairs, key=lambda x: x.conviction_score, reverse=True):
        ic_str = f"IC={a.direction_ic:+.3f}" if a.direction_ic is not None else "acc=0.586"
        vol_str = f"vol={a.estimated_vol_3d*100:.3f}%[{a.vol_source[:3]}]"
        print(
            f"  {a.pair}  {a.direction_label:<18}  tier={a.confidence_tier:<6}  "
            f"conv={a.conviction_score:.4f}  {ic_str}  "
            f"pos={a.position_size_pct:.2f}%  SL={a.sl_pct:.3f}%  TP={a.tp_pct:.3f}%  "
            f"R:R={a.risk_reward_ratio}  {vol_str}"
        )
    print(f"{'='*65}")


# Smoke test on three contrasting dates
for test_date in ["2024-01-15", "2025-03-20", "2024-06-10"]:
    rows = df[df["date"] == pd.Timestamp(test_date)]
    if rows.empty:
        print(f"No data for {test_date}")
        continue
    report = build_coordinator_report(rows)
    print_report(report)
    print()

CoordinatorReport  2024-01-15  |  regime=normal
  overall_action: TRADE
  top_pick: GBPUSD  |  ranked: ['GBPUSD', 'EURUSD', 'USDJPY', 'USDCHF']

  GBPUSD  LONG GBPUSD         tier=medium  conv=0.0690  IC=+0.115  pos=1.00%  SL=1.611%  TP=2.684%  R:R=1.67  vol=0.480%[geo]
  EURUSD  SHORT EURUSD        tier=low     conv=0.0180  IC=+0.060  pos=0.50%  SL=1.611%  TP=2.684%  R:R=1.67  vol=0.480%[geo]
  USDJPY  LONG USDJPY         tier=low     conv=0.0174  IC=+0.058  pos=0.50%  SL=1.611%  TP=2.684%  R:R=1.67  vol=0.480%[geo]
  USDCHF  SHORT USDCHF        tier=low     conv=0.0099  IC=+0.033  pos=0.50%  SL=1.611%  TP=2.684%  R:R=1.67  vol=0.480%[geo]

CoordinatorReport  2025-03-20  |  regime=high_attention
  overall_action: TRADE
  top_pick: GBPUSD  |  ranked: ['GBPUSD', 'EURUSD', 'USDJPY', 'USDCHF']

  GBPUSD  SHORT GBPUSD        tier=medium  conv=0.0690  IC=+0.115  pos=0.75%  SL=1.433%  TP=2.388%  R:R=1.67  vol=0.427%[geo]
  EURUSD  SHORT EURUSD        tier=low     conv=0.0180  IC=+0.060  pos=

## 7. Batch Run — Distribution Stats (2022–2025)

In [33]:
dates = sorted(df["date"].unique())
reports: list[CoordinatorReport] = []
for d in dates:
    rows = df[df["date"] == d]
    reports.append(build_coordinator_report(rows))

n = len(reports)
n_trade = sum(r.overall_action == "trade" for r in reports)
n_hold  = sum(r.overall_action == "hold"  for r in reports)
top_picks = [r.top_pick for r in reports if r.top_pick is not None]

print(f"Total dates: {n}")
print(f"  TRADE: {n_trade} ({n_trade/n:.1%})")
print(f"  HOLD:  {n_hold}  ({n_hold/n:.1%})")

print(f"\nTop pick distribution (when action=TRADE):")
from collections import Counter
for pair, cnt in Counter(top_picks).most_common():
    print(f"  {pair}: {cnt} ({cnt/len(top_picks):.1%})")

print(f"\nPosition size distribution (all pairs, trade days):")
all_pa = [pa for r in reports if r.overall_action == "trade" for pa in r.pairs]
pos_sizes = [a.position_size_pct for a in all_pa if a.direction is not None]
print(f"  mean={np.mean(pos_sizes):.3f}%  median={np.median(pos_sizes):.3f}%"
      f"  min={np.min(pos_sizes):.3f}%  max={np.max(pos_sizes):.3f}%")

print(f"\nSL/TP stats (all pairs, trade days):")
sls = [a.sl_pct for a in all_pa if a.direction is not None]
tps = [a.tp_pct for a in all_pa if a.direction is not None]
print(f"  SL: mean={np.mean(sls):.3f}%  p25={np.percentile(sls,25):.3f}%  p75={np.percentile(sls,75):.3f}%")
print(f"  TP: mean={np.mean(tps):.3f}%  p25={np.percentile(tps,25):.3f}%  p75={np.percentile(tps,75):.3f}%")

print(f"\nHOLD triggers:")
for r in reports:
    if r.overall_action == "hold":
        print(f"  {r.date.date()}: {r.hold_reason}")

Total dates: 1043
  TRADE: 1043 (100.0%)
  HOLD:  0  (0.0%)

Top pick distribution (when action=TRADE):
  GBPUSD: 869 (83.3%)
  USDJPY: 174 (16.7%)

Position size distribution (all pairs, trade days):
  mean=0.624%  median=0.500%  min=0.375%  max=2.000%

SL/TP stats (all pairs, trade days):
  SL: mean=1.508%  p25=1.420%  p75=1.598%
  TP: mean=2.514%  p25=2.366%  p75=2.663%

HOLD triggers:


## 8. LLM Prompt Template

The LLM receives `narrative_context` (pre-computed numbers) and produces **only prose** — no invented numbers. Numbers come exclusively from the coordinator.

## 8. LLM Prompt Template

In [34]:
def build_llm_prompt(report: CoordinatorReport) -> str:
    """
    Renders a structured prompt for the LLM.
    All numbers come from CoordinatorReport — LLM must not invent or modify any figure.
    LLM task: narrate rationale in clear, actionable language (3-6 sentences).
    """
    ctx = report.narrative_context
    lines = [
        "You are an FX market analyst assistant. Based on the quantitative analysis below,",
        "explain the recommendation in clear, actionable language.",
        "RULES: Do NOT invent numbers. Do NOT modify numbers. Do NOT add caveats not in the data.",
        "Mention confidence tier, signal source, risk/reward, and any relevant market context.",
        "",
        f"DATE: {ctx['date']}",
        f"GLOBAL REGIME: {ctx['global_regime']}",
        f"OVERALL ACTION: {ctx['overall_action'].upper()}",
    ]

    if ctx["overall_action"] == "hold":
        lines.append(f"HOLD REASON: {ctx['hold_reason']}")
        lines.append("")
        lines.append("TASK: In 2-3 sentences, explain why the system recommends staying out today.")
    else:
        rec = ctx["top_recommendation"]
        lines += [
            "",
            "── TOP RECOMMENDATION ──────────────────────────────────────",
            f"  Pair:            {rec['pair']}",
            f"  Action:          {rec['action']}",
            f"  Confidence:      {rec['confidence']} (source: {rec['signal_source']})",
            f"  Direction IC:    {rec['direction_ic'] if rec['direction_ic'] else 'N/A (validated by accuracy=58.6%)'}",
            f"  Horizon:         {rec['horizon']}",
            f"  Position size:   {rec['position_size_pct']}% of equity",
            f"  Stop loss:       {rec['sl_pct']}% below entry",
            f"  Take profit:     {rec['tp_pct']}% above entry",
            f"  Risk/reward:     {rec['risk_reward']}:1",
            f"  Expected vol:    {rec['estimated_vol_pct']}% daily (source: {rec['vol_source']})",
            "",
            "── ALL PAIRS (ranked by conviction) ────────────────────────",
        ]
        for p in ctx["all_pairs"]:
            stress = "⚠ stress" if p.get("composite_stress") else ""
            lines.append(
                f"  {p['pair']}: {p['action']:<5} ({p['confidence']}, conv={p['conviction']:.4f}, "
                f"vol={p['estimated_vol_pct']:.3f}%[{p['vol_source'][:3]}]) "
                f"driver={p['macro_driver']}  {stress}"
            )
        lines += [
            "",
            "── MARKET CONTEXT ──────────────────────────────────────────",
            f"  Geo regime:          {ctx['all_pairs'][0]['geo_risk_regime']}",
            f"  Macro attention z:   {ctx['all_pairs'][0]['macro_attention_z']:.3f}" if ctx["all_pairs"][0]["macro_attention_z"] else "  Macro attention z:   n/a",
            f"  GDELT tone z:        {ctx['all_pairs'][0]['gdelt_tone_z']:.3f}" if ctx["all_pairs"][0]["gdelt_tone_z"] else "  GDELT tone z:        n/a",
            "",
            "TASK: In 4-6 sentences explain:",
            "  1. What to do and why (primary signal source and its confidence)",
            "  2. How much to risk and where to place stop-loss / take-profit",
            "  3. Any regime or vol context worth highlighting",
            "  4. One-line summary of the secondary pairs (not recommended but notable)",
        ]

    return "\n".join(lines)


# Sample prompt for 2025-03-20 (high_attention day)
sample_rows = df[df["date"] == pd.Timestamp("2025-03-20")]
sample_report = build_coordinator_report(sample_rows)
print(build_llm_prompt(sample_report))

You are an FX market analyst assistant. Based on the quantitative analysis below,
explain the recommendation in clear, actionable language.
RULES: Do NOT invent numbers. Do NOT modify numbers. Do NOT add caveats not in the data.
Mention confidence tier, signal source, risk/reward, and any relevant market context.

DATE: 2025-03-20
GLOBAL REGIME: high_attention
OVERALL ACTION: TRADE

── TOP RECOMMENDATION ──────────────────────────────────────
  Pair:            GBPUSD
  Action:          SHORT
  Confidence:      medium (source: macro_gbpusd)
  Direction IC:    0.115
  Horizon:         5d
  Position size:   0.75% of equity
  Stop loss:       1.4328% below entry
  Take profit:     2.388% above entry
  Risk/reward:     1.67:1
  Expected vol:    0.427% daily (source: geo)

── ALL PAIRS (ranked by conviction) ────────────────────────
  GBPUSD: SHORT (medium, conv=0.0690, vol=0.427%[geo]) driver=fundamental_mispricing_score  ⚠ stress
  EURUSD: SHORT (low, conv=0.0180, vol=0.427%[geo]) driver=

## 9. Summary

In [35]:
print("=" * 70)
print("COORDINATOR NB04 — CoordinatorReport SUMMARY")
print("=" * 70)

print("""
DESIGN DECISIONS (all deterministic — LLM never synthesizes numbers)
─────────────────────────────────────────────────────────────────────

CONVICTION SCORE  =  tier_weight × direction_edge
  high    (USDJPY tech gate):  1.0 × 0.086  = 0.086   pos=2.0%
  medium  (GBPUSD macro):      0.6 × 0.115  = 0.069   pos=1.0%
  low     (other macro):       0.3 × 0.033–0.060       pos=0.5%
  high_attention regime:  all positions × 0.75

SL/TP FROM CALIBRATED VOL FORECAST
  estimated_vol_3d = OLS(vol_signal_norm → fwd_vol_3d, train 2022-2023)
    Geo:        IC test = +0.116  (2024-2025)
    StockTwits: IC test = +0.230  (2025 only)
  expected_move: vol_3d × 1 (1d horizon) or × √5 (5d horizon)
  SL = 1.5 × expected_move × 100  (% of entry)
  TP = 2.5 × expected_move × 100  → R:R always 1.67

HOLD GATE
  max(conviction) < 0.02             → hold unconditionally
  high_attention + conviction < 0.05 → hold
  In practice: GBPUSD medium (conv=0.069) keeps system in TRADE 100%
  of dates where macro_direction is available.  HOLD fires only on
  data-gap days (0 occurrences 2022-2025).

TOP PICK DISTRIBUTION (2022-2025)
  GBPUSD: 83.3% of days  (medium conviction, macro IC=+0.115)
  USDJPY: 16.7% of days  (high conviction, tech gate fires acc=58.6%)

TYPICAL TRADE PARAMETERS
  SL: 1.42–1.60% (p25–p75)   TP: 2.37–2.67%
  Position: 0.5% (low), 0.75–1.0% (medium), 1.5–2.0% (high)

LLM INTEGRATION — build_llm_prompt(report) outputs a structured
  briefing packet. LLM receives all numbers pre-computed and narrates:
  (1) what/why, (2) risk params, (3) regime context, (4) secondary pairs.

LIVE INFRA NOTES (for future implementation)
  • current_price needed for SL/TP in pips: pips = sl_pct/100 × price / pip_size
  • account_equity needed for position in currency: size = pos_pct/100 × equity
  • vol calibration constants (GEO_SLOPE/INTERCEPT, ST_SLOPE/INTERCEPT)
    should be re-fit periodically (recommend: quarterly expanding window)
""")

print("Next: NB05 — Backtester (P&L simulation using CoordinatorReport signals)")
print("=" * 70)

COORDINATOR NB04 — CoordinatorReport SUMMARY

DESIGN DECISIONS (all deterministic — LLM never synthesizes numbers)
─────────────────────────────────────────────────────────────────────

CONVICTION SCORE  =  tier_weight × direction_edge
  high    (USDJPY tech gate):  1.0 × 0.086  = 0.086   pos=2.0%
  medium  (GBPUSD macro):      0.6 × 0.115  = 0.069   pos=1.0%
  low     (other macro):       0.3 × 0.033–0.060       pos=0.5%
  high_attention regime:  all positions × 0.75

SL/TP FROM CALIBRATED VOL FORECAST
  estimated_vol_3d = OLS(vol_signal_norm → fwd_vol_3d, train 2022-2023)
    Geo:        IC test = +0.116  (2024-2025)
    StockTwits: IC test = +0.230  (2025 only)
  expected_move: vol_3d × 1 (1d horizon) or × √5 (5d horizon)
  SL = 1.5 × expected_move × 100  (% of entry)
  TP = 2.5 × expected_move × 100  → R:R always 1.67

HOLD GATE
  max(conviction) < 0.02             → hold unconditionally
  high_attention + conviction < 0.05 → hold
  In practice: GBPUSD medium (conv=0.069) keeps sys